Inverter Tagging - Capela
=
This notebook numbers the inverters of UFV Capela inside each `PRK.UFV.PST` group (top-to-bottom, left-to-right). It writes an AutoCAD script (`.scr`) that re-inserts every `EMF379_inversor` block with its `PRK`, `UFV`, `PST`, `INV`, power and `GRP` attributes.

**Workflow overview:**
1. Read the inverter list from Excel, reorder the columns, set the power and build `GRP` = `PRK.UFV.PST`.
2. Sort the inverters by group, then top-to-bottom / left-to-right.
3. Number them inside each group and build the insert commands.
4. Write `Capela - Tagueamento Inversores.scr`.

**Input prerequisites** (the code assumes all of these, and it doesn't check them):
- **File:** `inversores capela.xlsx` in `C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Tagueamento Inversor e Estruturas\` (hardcoded).
- **Exactly 11 columns in this order:** `DSJ`, `GRP`, `INV`, `Layer`, `Position X`, `Position Y`, `PST`, `QBT`, `UFV`, `POT`, `PRK`. The code reorders them **by position** (`iloc`), so a different column order silently mislabels the data.
  - Only `Layer`, `Position X`, `Position Y`, `PRK`, `UFV` and `PST` are used. `GRP`, `INV` and `POT` are recalculated or overwritten.
- `PRK`, `UFV` and `PST` should be single digits: `GRP` is sorted as text, so `1.10.1` would sort before `1.2.1`.
- The input has no rotation: every inverter is re-inserted at 0°.

**AutoCAD prerequisites for the generated script:**
- The block `EMF379_inversor` is defined in the drawing with *Scale uniformly* on (the script answers one scale factor, `0.001`).
- Its attributes are filled positionally: `PRK`, `UFV`, `PST`, `INV`, `-`, `-`, `POT`, `GRP`.
- Every `Layer` value exists in the drawing.
- Run the `.scr` with `ATTREQ = 1` and `ATTDIA = 0`. The script only inserts: delete the original inverter blocks first, or they end up duplicated.

Loading and preparing the inverters
=
- Reads the Excel file and reorders its columns by position to `Layer`, `Position X`, `Position Y`, `GRP`, `PRK`, `UFV`, `PST`, `INV`, `QBT`, `DSJ`, `POT` (see the input prerequisites).
- `POT` = `730` for every inverter (hardcoded power value; change it per project).
- `GRP` = `PRK.UFV.PST` (e.g. `1.2.1`). It overwrites the `GRP` column of the file and is the group inside which the inverters are numbered.
- `numpy` is imported but not used.

In [49]:
import pandas as pd
import numpy as np

inversores_df = pd.read_excel(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Tagueamento Inversor e Estruturas\inversores capela.xlsx")
inversores_df = inversores_df.iloc[:,[3,4,5,1,10,8,6,2,7,0,9]] #Reordering the columns
inversores_df["POT"] = 730
inversores_df["GRP"] = inversores_df["PRK"].astype(str) + "." + inversores_df["UFV"].astype(str) + "." + inversores_df["PST"].astype(str)

print(inversores_df)

                Layer  Position X   Position Y    GRP  PRK  UFV  PST   INV  \
0    EMF_em_pst_1.2.1 -18967.4361  156467.4556  1.2.1    1    2    1   NaN   
1    EMF_em_pst_2.1.2 -18147.5526  158119.4556  2.1.2    2    1    2  15.0   
2    EMF_em_pst_2.1.2 -18385.7137  158272.4556  2.1.2    2    1    2   2.0   
3    EMF_em_pst_2.1.2 -18391.9890  158236.4556  2.1.2    2    1    2   5.0   
4    EMF_em_pst_2.1.2 -18387.7374  158191.4556  2.1.2    2    1    2   9.0   
..                ...         ...          ...    ...  ...  ...  ...   ...   
181  EMF_em_pst_2.1.1 -18223.7624  157705.4556  2.1.1    2    1    1   NaN   
182  EMF_em_pst_2.1.1 -17771.1705  157687.4556  2.1.1    2    1    1   NaN   
183  EMF_em_pst_2.1.1 -17816.8082  157669.4556  2.1.1    2    1    1   NaN   
184  EMF_em_pst_2.1.1 -18176.4794  157777.4556  2.1.1    2    1    1   NaN   
185  EMF_em_pst_1.2.2 -19489.0700  156652.4556  1.2.2    1    2    2   NaN   

     QBT  DSJ  POT  
0    NaN  NaN  730  
1    NaN  NaN  730  


Sorting
=
Sort order: `GRP` ascending (as text), then `Position Y` descending, then `Position X` ascending (top-to-bottom, left-to-right inside each group). `ignore_index=True` renumbers the rows so the numbering in the next step follows this order.

In [50]:
inversores_df.sort_values(by=["GRP", "Position Y", "Position X"], ascending=[True, False, True], inplace=True, ignore_index=True)
print(inversores_df)

                Layer  Position X   Position Y    GRP  PRK  UFV  PST   INV  \
0    EMF_em_pst_1.1.1 -18215.3295  157687.4556  1.1.1    1    1    1   NaN   
1    EMF_em_pst_1.1.1 -18255.1413  157669.4556  1.1.1    1    1    1   NaN   
2    EMF_em_pst_1.1.1 -18207.1750  157651.4556  1.1.1    1    1    1   NaN   
3    EMF_em_pst_1.1.1 -18250.5099  157633.4556  1.1.1    1    1    1   NaN   
4    EMF_em_pst_1.1.1 -18230.4224  157606.4556  1.1.1    1    1    1   NaN   
..                ...         ...          ...    ...  ...  ...  ...   ...   
181  EMF_em_pst_2.3.1 -18839.6876  157894.4556  2.3.1    2    3    1  11.0   
182  EMF_em_pst_2.3.1 -18839.6876  157876.4556  2.3.1    2    3    1  12.0   
183  EMF_em_pst_2.3.1 -18753.5067  157871.4328  2.3.1    2    3    1  13.0   
184  EMF_em_pst_2.3.1 -18839.6876  157858.4556  2.3.1    2    3    1  14.0   
185  EMF_em_pst_2.3.1 -18811.3226  157826.4328  2.3.1    2    3    1  15.0   

     QBT  DSJ  POT  
0    NaN  NaN  730  
1    NaN  NaN  730  


Numbering and AutoCAD commands
=
- `INV` = running number inside each `GRP` (`cumcount() + 1`). It replaces the values from the file (many were empty).
- `Script` builds two AutoLISP lines per inverter:
  `(command-s "._layer" "set" "<Layer>" "")`
  `(command-s "._insert" "EMF379_inversor" "X,Y" 0.001 0 PRK UFV PST INV "-" "-" POT "GRP")`
  - `0.001` is the single (uniform) scale factor and `0` the rotation. The scale depends on the block definition; keep it in sync if the block changes.
  - `PRK`, `UFV`, `PST` and `INV` are zero-padded to 2 digits. `GRP` keeps the unpadded form (`1.2.1`).

In [51]:
inversores_df["INV"] = inversores_df.groupby("GRP").cumcount()+1
inversores_df["Script"] = inversores_df.apply(
    lambda row: (
        f'(command-s "._layer" "set" "{row["Layer"]}" "")\n'
        f'(command-s "._insert" "EMF379_inversor" '
        f'"{row["Position X"]},{row["Position Y"]}" 0.001 0 {str(row["PRK"]).zfill(2)} {str(row["UFV"]).zfill(2)} {str(row["PST"]).zfill(2)} {str(row["INV"]).zfill(2)} "-" "-" {row["POT"]} "{row["GRP"]}")'
    ),
    axis=1
)
print(inversores_df)

                Layer  Position X   Position Y    GRP  PRK  UFV  PST  INV  \
0    EMF_em_pst_1.1.1 -18215.3295  157687.4556  1.1.1    1    1    1    1   
1    EMF_em_pst_1.1.1 -18255.1413  157669.4556  1.1.1    1    1    1    2   
2    EMF_em_pst_1.1.1 -18207.1750  157651.4556  1.1.1    1    1    1    3   
3    EMF_em_pst_1.1.1 -18250.5099  157633.4556  1.1.1    1    1    1    4   
4    EMF_em_pst_1.1.1 -18230.4224  157606.4556  1.1.1    1    1    1    5   
..                ...         ...          ...    ...  ...  ...  ...  ...   
181  EMF_em_pst_2.3.1 -18839.6876  157894.4556  2.3.1    2    3    1   11   
182  EMF_em_pst_2.3.1 -18839.6876  157876.4556  2.3.1    2    3    1   12   
183  EMF_em_pst_2.3.1 -18753.5067  157871.4328  2.3.1    2    3    1   13   
184  EMF_em_pst_2.3.1 -18839.6876  157858.4556  2.3.1    2    3    1   14   
185  EMF_em_pst_2.3.1 -18811.3226  157826.4328  2.3.1    2    3    1   15   

     QBT  DSJ  POT                                             Script  
0  

Writing the script
=
Writes every `Script` line to `Capela - Tagueamento Inversores.scr` in the `Tagueamento Inversor e Estruturas` folder. Run it in AutoCAD with the `SCRIPT` command after deleting the original inverter blocks.

In [52]:
with open (r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Tagueamento Inversor e Estruturas\Capela - Tagueamento Inversores.scr", 'w') as f:
    f.write('\n'.join(inversores_df["Script"]) + '\n')